In [34]:
import requests
import numpy as np
import matplotlib.pyplot as plt

In [36]:
# 1. Запит до Open-Elevation API 
url = "https://api.open-elevation.com/api/v1/lookup"

data = {
    "locations": [
        {"latitude": 48.164214, "longitude": 24.536044},
        {"latitude": 48.164983, "longitude": 24.534836},
        {"latitude": 48.165605, "longitude": 24.534068},
        {"latitude": 48.166228, "longitude": 24.532915},
        {"latitude": 48.166777, "longitude": 24.531927},
        {"latitude": 48.167326, "longitude": 24.530884},
        {"latitude": 48.167011, "longitude": 24.530061},
        {"latitude": 48.166053, "longitude": 24.528039},
        {"latitude": 48.166655, "longitude": 24.526064},
        {"latitude": 48.166497, "longitude": 24.523574},
        {"latitude": 48.166128, "longitude": 24.520214},
        {"latitude": 48.165416, "longitude": 24.517170},
        {"latitude": 48.164546, "longitude": 24.514640},
        {"latitude": 48.163412, "longitude": 24.512980},
        {"latitude": 48.162331, "longitude": 24.511715},
        {"latitude": 48.162015, "longitude": 24.509462},
        {"latitude": 48.162147, "longitude": 24.506932},
        {"latitude": 48.161751, "longitude": 24.504244},
        {"latitude": 48.161197, "longitude": 24.501793},
        {"latitude": 48.160580, "longitude": 24.500537},
        {"latitude": 48.160250, "longitude": 24.500106},
    ]
}

response = requests.post(url, json=data)
print(response.status_code)

if response.status_code == 200:
    data = response.json()
    results = data["results"]
    n = len(results)
    print(f"Отримано вузлів: {n}") 
else:
    print(f"Помилка сервера: {response.status_code}")

504
Помилка сервера: 504


In [30]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000  # Радіус Землі
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
    return 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1-a))

coords = [(p["latitude"], p["longitude"]) for p in results]
elevations = [p["elevation"] for p in results]

distances = [0]
for i in range(1, n):
    d = haversine(*coords[i-1], *coords[i])
    distances.append(distances[-1] + d)

print(f"Загальна відстань маршруту: {distances[-1]:.2f} м")

NameError: name 'results' is not defined

In [12]:
def solve_spline(x, y):
    n_pts = len(x) - 1
    h = np.diff(x)
    
    # Система рівнянь для c_i
    alpha = np.zeros(n_pts + 1)
    beta = np.ones(n_pts + 1)
    gamma = np.zeros(n_pts + 1)
    delta = np.zeros(n_pts + 1)
    
    for i in range(1, n_pts):
        alpha[i] = h[i-1]
        beta[i] = 2 * (h[i-1] + h[i])
        gamma[i] = h[i]
        delta[i] = 3 * ((y[i+1] - y[i])/h[i] - (y[i] - y[i-1])/h[i-1])
    
    # Пряма прогонка (Forward elimination)
    A_p = np.zeros(n_pts + 1)
    B_p = np.zeros(n_pts + 1)
    for i in range(1, n_pts):
        m = alpha[i] * A_p[i-1] + beta[i]
        A_p[i] = -gamma[i] / m
        B_p[i] = (delta[i] - alpha[i] * B_p[i-1]) / m
        
    # Зворотна прогонка (Backward substitution)
    c_coeffs = np.zeros(n_pts + 1)
    for i in range(n_pts - 1, 0, -1):
        c_coeffs[i] = A_p[i] * c_coeffs[i+1] + B_p[i]
        
    # Розрахунок інших коефіцієнтів за формулами з методички
    a_coeffs = y[:-1]
    d_coeffs = np.diff(c_coeffs) / (3 * h)
    b_coeffs = (np.diff(y) / h) - (h / 3) * (c_coeffs[1:] + 2 * c_coeffs[:-1])
    
    return a_coeffs, b_coeffs, c_coeffs[:-1], d_coeffs

# Обчислення
a, b, c, d = solve_spline(np.array(distances), np.array(elevations))
print("Коефіцієнти сплайнів успішно знайдені.")

NameError: name 'distances' is not defined

In [33]:
# Побудова графіка
plt.figure(figsize=(12, 6))
plt.scatter(distances, elevations, color='red', label='Вузли API')

# Генерація точок для гладкої лінії
x_fine = np.linspace(distances[0], distances[-1], 500)
y_fine = []
for val in x_fine:
    idx = np.searchsorted(distances, val) - 1
    idx = max(0, min(idx, len(distances) - 2))
    dx = val - distances[idx]
    y_val = a[idx] + b[idx]*dx + c[idx]*dx**2 + d[idx]*dx**3
    y_fine.append(y_val)

plt.plot(x_fine, y_fine, 'b-', label='Кубічний сплайн (20 вузлів)')
plt.title("Профіль маршруту: Заросляк — Говерла")
plt.xlabel("Відстань (м)")
plt.ylabel("Висота (м)")
plt.legend()
plt.grid(True)
plt.show()

# Додаткове завдання
total_ascent = sum(max(elevations[i]-elevations[i-1], 0) for i in range(1, len(elevations)))
mass = 80
g = 9.81
energy_kj = (mass * g * total_ascent) / 1000

print(f"Сумарний набір висоти: {total_ascent} м")
print(f"Механічна робота: {energy_kj:.2f} кДж")
print(f"Енергія в ккал: {energy_kj / 4.184:.2f} ккал")

NameError: name 'distances' is not defined

<Figure size 1200x600 with 0 Axes>